# 00 Effective Area

任意の時刻の選手座標から、守備チームと攻撃チームの Effective Area（EA）を計算する。

## アルゴリズム（Definition 4 / 表面積指標 2）

PDFの手書き疑似コードに合わせた定義。

### 守備 EA
1. 守備チーム選手座標から 11C3 = 165 個の三角形を生成する（GK含む11人）
2. 各三角形の周囲長を計算する
3. **周囲長が36m未満の三角形だけ**を採用する
4. 採用した守備三角形の和集合が守備 EA

### 攻撃 EA
1. 攻撃チーム選手座標から同様に 165 個の三角形を生成する
2. 攻撃EAを空集合から始める
3. 各攻撃三角形について、**守備EA + 採用済み攻撃EA**と面積を持って重なる場合は除外する
4. 重ならない三角形だけを順番に採用する
5. 採用した攻撃三角形の和集合が攻撃 EA

## 座標系

トラッキングXMLの座標（0〜1 正規化）をメートルに変換して使用する。
```
x_m = x_raw * pitch_width   (例: x_raw * 105)
y_m = y_raw * pitch_height  (例: y_raw * 68)
```
ピッチサイズは metadata XML から自動取得する。


## 設定

新しい試合を分析する場合は `MATCH_DATE`、`MATCH_ID`、`MATCH_LABEL` を変更する。

`INCLUDE_GK` は論文確認後に設定する。

In [ ]:
import ast
import xml.etree.ElementTree as ET
from itertools import combinations
from pathlib import Path

import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from shapely.geometry import MultiPolygon, Polygon
from shapely.ops import unary_union

def find_project_root(start=None):
    current = Path.cwd().resolve() if start is None else Path(start).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / 'data').exists() and (candidate / 'はるた').exists():
            return candidate
    raise FileNotFoundError('プロジェクトルートが見つかりません。data/ と はるた/ がある場所で実行してください。')


PROJECT_ROOT = find_project_root()
HARUTA_ROOT = PROJECT_ROOT / 'はるた'

MATCH_DATE = '2023-11-25'
MATCH_ID = '117093'
MATCH_LABEL = 'tsukuba_vs_tsukuba-b'
DATASET_NAME = f'{MATCH_DATE}_{MATCH_ID}_{MATCH_LABEL}'

DATASET_DIR = PROJECT_ROOT / 'data' / 'candidates' / '01_sorted' / DATASET_NAME
TRACKING_XML = DATASET_DIR / '01_tracking_core' / f'{MATCH_ID}_tracker_box_data.xml'
METADATA_XML = DATASET_DIR / '03_player_master' / f'{MATCH_ID}_tracker_box_metadata.xml'
OUTPUT_DIR = HARUTA_ROOT / 'outputs' / 'tables' / DATASET_NAME
FIGURE_DIR = HARUTA_ROOT / 'outputs' / 'figures' / DATASET_NAME / 'effective_area'

# Effective Area パラメータ
PERIMETER_THRESHOLD = 36.0  # meters: 守備三角形の採用閾値（周囲長が36m未満の三角形を採用）
INCLUDE_GK = True           # 論文通り GK を含む 11 人（11C3 = 165 三角形）
VALID_BALL_STATUSES = ('HOME', 'AWAY')

print(DATASET_NAME)
print('tracking XML exists:', TRACKING_XML.exists())
print('metadata XML exists:', METADATA_XML.exists())


## 関数定義

In [ ]:
def load_tracking_metadata(metadata_xml_path):
    """metadata XML からピッチサイズ、チーム情報、選手情報を読む。"""
    root = ET.parse(metadata_xml_path).getroot()

    pitch = root.find('pitch')
    pitch_info = {
        'width': float(pitch.attrib['width']),
        'height': float(pitch.attrib['height']),
    }

    teams = []
    for team in root.findall('.//teams/team'):
        teams.append({
            'team_id': str(team.attrib['id']),
            'team_name': team.attrib['name'],
            'side': team.attrib['side'],
            'team_label': 'A' if team.attrib['side'] == 'away' else 'B',
        })

    players = []
    for player in root.findall('.//players/player'):
        players.append({
            'player_id': str(player.attrib['id']),
            'player_name': player.attrib.get('name', ''),
            'position': player.attrib.get('position', ''),
            'team_id': str(player.attrib['teamId']),
        })

    teams_df = pd.DataFrame(teams)
    players_df = pd.DataFrame(players).merge(
        teams_df[['team_id', 'side', 'team_label']],
        on='team_id',
        how='left',
    )
    return {'pitch': pitch_info, 'teams': teams_df, 'players': players_df}


In [ ]:
def load_player_positions_by_frame(tracking_xml_path, metadata, include_gk=True):
    """
    tracking XML から全フレームの選手座標をメートル単位で読む。

    Returns: list of dicts
        frame_number, match_time, event_period, ball_status,
        team_positions: {team_id: [(x_m, y_m), ...]}
    """
    players_by_id = metadata['players'].set_index('player_id').to_dict('index')
    pitch_w = metadata['pitch']['width']
    pitch_h = metadata['pitch']['height']

    frames = []
    for _, elem in ET.iterparse(str(tracking_xml_path), events=('end',)):
        if elem.tag != 'frame':
            continue

        team_positions = {}
        for player in elem.findall('player'):
            player_id = str(player.attrib['playerId'])
            player_info = players_by_id.get(player_id)
            if not player_info:
                continue
            if not include_gk and player_info.get('position') == 'GK':
                continue

            team_id = str(player_info['team_id'])
            raw_x, raw_y = ast.literal_eval(player.attrib['loc'])
            x_m = float(raw_x) * pitch_w
            y_m = float(raw_y) * pitch_h
            team_positions.setdefault(team_id, []).append((x_m, y_m))

        frames.append({
            'frame_number': int(float(elem.attrib['frameNumber'])),
            'match_time': int(float(elem.attrib['matchTime'])),
            'event_period': elem.attrib.get('eventPeriod', ''),
            'ball_status': elem.attrib.get('ballStatus', ''),
            'team_positions': team_positions,
        })
        elem.clear()

    return frames


In [ ]:
def _perimeter(p1, p2, p3):
    """3点間の周囲長をメートルで返す。"""
    d = lambda a, b: np.sqrt((a[0] - b[0]) ** 2 + (a[1] - b[1]) ** 2)
    return d(p1, p2) + d(p2, p3) + d(p3, p1)


def compute_defensive_ea(player_positions, perimeter_threshold=36.0):
    """
    守備 Effective Area を計算する。

    PDFの定義に合わせ、守備側11人から作れる三角形のうち、
    周囲長が perimeter_threshold 未満の三角形だけを採用し、
    それらの和集合を守備EAとする。

    player_positions: list of (x, y) in meters
    Returns: (Shapely geometry, n_adopted_triangles)
    """
    adopted = []
    for i, j, k in combinations(range(len(player_positions)), 3):
        p1, p2, p3 = player_positions[i], player_positions[j], player_positions[k]
        tri = Polygon([p1, p2, p3])
        if not tri.is_valid or tri.area <= 1e-9:
            continue
        if _perimeter(p1, p2, p3) >= perimeter_threshold:
            continue
        adopted.append(tri)
    if not adopted:
        return Polygon(), 0
    return unary_union(adopted), len(adopted)


def compute_attacking_ea(player_positions, defensive_ea):
    """
    攻撃 Effective Area を計算する。

    PDFの定義に合わせ、攻撃側三角形が現在までの攻撃EAと
    守備EAの和集合に重ならない場合だけ採用する。

    player_positions: list of (x, y) in meters
    defensive_ea: Shapely geometry（compute_defensive_ea の戻り値）
    Returns: (Shapely geometry, n_adopted_triangles)
    """
    attacking_ea = Polygon()
    n_adopted = 0
    for i, j, k in combinations(range(len(player_positions)), 3):
        tri = Polygon([player_positions[i], player_positions[j], player_positions[k]])
        if not tri.is_valid or tri.area < 1e-9:
            continue
        occupied = attacking_ea.union(defensive_ea)
        if occupied.intersection(tri).area >= 1e-9:
            continue
        attacking_ea = attacking_ea.union(tri)
        n_adopted += 1
    return attacking_ea, n_adopted


def compute_effective_area(def_positions, att_positions, perimeter_threshold=36.0):
    """
    守備・攻撃 EA をまとめて計算する。

    def_positions: list of (x, y) in meters（守備チーム）
    att_positions: list of (x, y) in meters（攻撃チーム）
    Returns: dict
    """
    defensive_ea, n_def = compute_defensive_ea(def_positions, perimeter_threshold)
    attacking_ea, n_att = compute_attacking_ea(att_positions, defensive_ea)
    return {
        'defensive_ea': defensive_ea,
        'attacking_ea': attacking_ea,
        'defensive_area_m2': defensive_ea.area,
        'attacking_area_m2': attacking_ea.area,
        'n_defensive_triangles': n_def,
        'n_attacking_triangles': n_att,
    }



In [ ]:
def _draw_geom(ax, geom, facecolor, edgecolor, alpha, linewidth=1.0):
    """Shapely Polygon / MultiPolygon をピッチ上に描画する。"""
    if geom.is_empty:
        return
    geoms = list(geom.geoms) if isinstance(geom, MultiPolygon) else [geom]
    for poly in geoms:
        x, y = poly.exterior.xy
        ax.fill(x, y, facecolor=facecolor, edgecolor=edgecolor,
                alpha=alpha, linewidth=linewidth)


def plot_effective_area(frame_data, metadata, ea_result,
                        def_team_id, att_team_id, save_path=None):
    """
    ピッチ上に守備EA（青）と攻撃EA（赤）を描画する。

    frame_data: load_player_positions_by_frame の1要素
    metadata: load_tracking_metadata の戻り値
    ea_result: compute_effective_area の戻り値
    """
    pitch_w = metadata['pitch']['width']
    pitch_h = metadata['pitch']['height']
    teams_idx = metadata['teams'].set_index('team_id')

    def_name = teams_idx.loc[def_team_id, 'team_name'] if def_team_id in teams_idx.index else def_team_id
    att_name = teams_idx.loc[att_team_id, 'team_name'] if att_team_id in teams_idx.index else att_team_id

    fig, ax = plt.subplots(figsize=(13, 9))
    ax.set_facecolor('#3d7a4a')

    # ピッチ
    pitch_rect = plt.Rectangle((0, 0), pitch_w, pitch_h,
                                facecolor='#4a8c56', edgecolor='white', linewidth=2)
    ax.add_patch(pitch_rect)
    ax.plot([pitch_w / 2, pitch_w / 2], [0, pitch_h], 'white', linewidth=1)
    center_circle = mpatches.Circle((pitch_w / 2, pitch_h / 2), 9.15,
                                    fill=False, color='white', linewidth=1)
    ax.add_patch(center_circle)

    # 守備 EA（青）
    _draw_geom(ax, ea_result['defensive_ea'],
               facecolor='#3366cc', edgecolor='#1144aa', alpha=0.45)

    # 攻撃 EA（赤）
    _draw_geom(ax, ea_result['attacking_ea'],
               facecolor='#cc3333', edgecolor='#aa1111', alpha=0.45)

    # 選手プロット
    pos = frame_data['team_positions']
    for x, y in pos.get(def_team_id, []):
        ax.plot(x, y, 'o', color='#3366cc', markersize=9,
                markeredgecolor='white', markeredgewidth=1.2)
    for x, y in pos.get(att_team_id, []):
        ax.plot(x, y, 'o', color='#cc3333', markersize=9,
                markeredgecolor='white', markeredgewidth=1.2)

    # 凡例
    legend_elements = [
        mpatches.Patch(facecolor='#3366cc', alpha=0.6,
                       label=f'{def_name}（守備）EA: {ea_result["defensive_area_m2"]:.1f} m\u00b2'),
        mpatches.Patch(facecolor='#cc3333', alpha=0.6,
                       label=f'{att_name}（攻撃）EA: {ea_result["attacking_area_m2"]:.1f} m\u00b2'),
    ]
    ax.legend(handles=legend_elements, loc='upper right', fontsize=10)

    match_time_s = frame_data['match_time'] / 1000
    ax.set_title(
        f'Effective Area — Frame {frame_data["frame_number"]}'
        f' ({frame_data["event_period"]}, {match_time_s:.1f}s,'
        f' ballStatus={frame_data["ball_status"]})',
        fontsize=12,
    )
    ax.set_xlim(-2, pitch_w + 2)
    ax.set_ylim(-2, pitch_h + 2)
    ax.set_aspect('equal')
    ax.set_xlabel('x (m)')
    ax.set_ylabel('y (m)')

    plt.tight_layout()
    if save_path:
        Path(save_path).parent.mkdir(parents=True, exist_ok=True)
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        print(f'Saved: {save_path}')
    plt.show()
    return fig, ax


In [ ]:
def process_all_frames(frames, metadata, perimeter_threshold=36.0):
    """
    フレームリストを受け取り、全フレームの EA を計算して DataFrame で返す。
    ball_status が HOME または AWAY のフレームのみ対象。
    """
    teams_by_side = metadata['teams'].set_index('side')
    home_id = teams_by_side.loc['home', 'team_id']
    away_id = teams_by_side.loc['away', 'team_id']
    teams_by_id = metadata['teams'].set_index('team_id')

    rows = []
    for frame in frames:
        ball_status = frame['ball_status']
        if ball_status not in ('HOME', 'AWAY'):
            continue

        pos = frame['team_positions']
        if home_id not in pos or away_id not in pos:
            continue

        att_id = home_id if ball_status == 'HOME' else away_id
        def_id = away_id if ball_status == 'HOME' else home_id

        att_pos = pos[att_id]
        def_pos = pos[def_id]
        if not att_pos or not def_pos:
            continue

        result = compute_effective_area(def_pos, att_pos, perimeter_threshold)

        att_label = teams_by_id.loc[att_id, 'team_label'] if att_id in teams_by_id.index else '?'
        def_label = teams_by_id.loc[def_id, 'team_label'] if def_id in teams_by_id.index else '?'

        rows.append({
            'frame_number': frame['frame_number'],
            'match_time_ms': frame['match_time'],
            'match_time_s': frame['match_time'] / 1000,
            'event_period': frame['event_period'],
            'ball_status': ball_status,
            'attacking_team_id': att_id,
            'defending_team_id': def_id,
            'attacking_team_label': att_label,
            'defending_team_label': def_label,
            'n_att_players': len(att_pos),
            'n_def_players': len(def_pos),
            'defensive_ea_m2': result['defensive_area_m2'],
            'attacking_ea_m2': result['attacking_area_m2'],
            'n_defensive_triangles': result['n_defensive_triangles'],
            'n_attacking_triangles': result['n_attacking_triangles'],
        })

    return pd.DataFrame(rows)


## メタデータ確認

In [ ]:
metadata = load_tracking_metadata(METADATA_XML)
print('Pitch:', metadata['pitch'])
print()
metadata['teams']


In [ ]:
metadata['players'].groupby(['team_label', 'position']).size().reset_index(name='count')


## トラッキングデータ読み込み

全フレームの選手座標をメートル単位で読む。

In [ ]:
print('Loading tracking data...')
frames = load_player_positions_by_frame(TRACKING_XML, metadata, include_gk=INCLUDE_GK)
print(f'Total frames loaded: {len(frames):,}')

valid_frames = [f for f in frames if f['ball_status'] in VALID_BALL_STATUSES]
print(f'Frames with valid ball status: {len(valid_frames):,}')
print()

# ball_status の分布
from collections import Counter
status_counts = Counter(f['ball_status'] for f in frames)
for status, count in sorted(status_counts.items()):
    print(f'  {status}: {count:,} frames')


## 単一フレームデモ

`DEMO_FRAME_INDEX` で有効フレームリストの何番目を使うかを指定する（0 = 最初の有効フレーム）。

In [ ]:
DEMO_FRAME_INDEX = 0  # valid_frames の何番目を使うか

demo_frame = valid_frames[DEMO_FRAME_INDEX]
print(f'Frame number : {demo_frame["frame_number"]}')
print(f'match_time   : {demo_frame["match_time"] / 1000:.1f}s')
print(f'event_period : {demo_frame["event_period"]}')
print(f'ball_status  : {demo_frame["ball_status"]}')
print()
for tid, positions in demo_frame['team_positions'].items():
    team_row = metadata['teams'][metadata['teams']['team_id'] == tid]
    label = team_row['team_label'].values[0] if not team_row.empty else tid
    name = team_row['team_name'].values[0] if not team_row.empty else tid
    print(f'  Team {label} ({name}): {len(positions)} players')


In [ ]:
# 攻守の割り当て
teams_by_side = metadata['teams'].set_index('side')
home_team_id = teams_by_side.loc['home', 'team_id']
away_team_id = teams_by_side.loc['away', 'team_id']

ball_status = demo_frame['ball_status']
att_team_id = home_team_id if ball_status == 'HOME' else away_team_id
def_team_id = away_team_id if ball_status == 'HOME' else home_team_id

att_positions = demo_frame['team_positions'].get(att_team_id, [])
def_positions = demo_frame['team_positions'].get(def_team_id, [])

print(f'Attacking team: {att_team_id} ({len(att_positions)} players)')
print(f'Defending team: {def_team_id} ({len(def_positions)} players)')


In [ ]:
ea_result = compute_effective_area(def_positions, att_positions, PERIMETER_THRESHOLD)

print('=== Effective Area Results ===')
print(f'Defensive EA : {ea_result["defensive_area_m2"]:8.2f} m\u00b2  (valid triangles: {ea_result["n_defensive_triangles"]})')
print(f'Attacking EA : {ea_result["attacking_area_m2"]:8.2f} m\u00b2  (adopted triangles: {ea_result["n_attacking_triangles"]})')


In [ ]:
fig, ax = plot_effective_area(
    demo_frame, metadata, ea_result,
    def_team_id=def_team_id,
    att_team_id=att_team_id,
    save_path=FIGURE_DIR / f'frame_{demo_frame["frame_number"]}.png',
)


## 全フレーム集計（オプション）

165 三角形 × フレーム数 の処理になるため、試合全体で数分かかる場合がある。

`SAMPLE_EVERY_N_FRAMES` を大きくすれば速くなる（例: 25 = 約1秒おきにサンプリング）。

In [ ]:
SAMPLE_EVERY_N_FRAMES = 25  # 有効フレームの中から N フレームおきにサンプリング

sampled = [f for i, f in enumerate(valid_frames) if i % SAMPLE_EVERY_N_FRAMES == 0]
print(f'Processing {len(sampled)} frames (sampled 1/{SAMPLE_EVERY_N_FRAMES})...')

ea_df = process_all_frames(sampled, metadata, PERIMETER_THRESHOLD)
print(f'Done. {len(ea_df)} rows')
ea_df.head()


In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
output_path = OUTPUT_DIR / 'effective_area.csv'
ea_df.to_csv(output_path, index=False)
print(f'Saved: {output_path}')
print()

print('Summary by attacking team:')
ea_df.groupby('attacking_team_label').agg(
    frames=('frame_number', 'count'),
    mean_def_ea_m2=('defensive_ea_m2', 'mean'),
    mean_att_ea_m2=('attacking_ea_m2', 'mean'),
    median_def_ea_m2=('defensive_ea_m2', 'median'),
    median_att_ea_m2=('attacking_ea_m2', 'median'),
).round(2)
